## The State of Tax Justice: Impute Missing Data
- Author: Alison Schultz, based on Javier Garcia Bernado's work
- Created: 20 August 2023
- Last updated: 1 September 2023

**Description**
- This notebook is one out of three notebooks to estimate the tax losses caused by profit shifting by multinational enterprises (MNEs). The analysis used the misalignment method based on the country-by-country reports (CbCR) published by the OECD.
- This notebook uses the extended CBCR dataset "data/intermediate/imputation_sample.csv" created in the notebook "1_clean" and imputes missing values, resulting in the notebook "data/final/cbcr_main.csv". It also generates all descriptives used to assess the imputation success.

**Outline**
1. 

**To dos before running this notebook**
1.

In [ ]:
import pandas as pd
import numpy as np
from config_aug import *
import tjn_tools
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from itertools import permutations, product
from sklearn.metrics import r2_score,make_scorer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LassoCV, Lasso, LinearRegression, BayesianRidge
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.model_selection import cross_validate, train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.inspection import permutation_importance

## 3. Imputate missing values

### 3.1 Import and define relevant datasets

In [ ]:
imputation_sample = pd.read_csv(f'{data_intermediate}/imputation_sample_2023.csv')
cbcr_main_no_imputation_allsubgroupsonly = pd.read_csv(f'{data_intermediate}/cbcr_main_no_imputation_allsubgroupsonly_2023.csv')

In [ ]:
# For the domestic imputation, we only use variables that are not bilateral in nature
domestic_variables = [
    'iso_parent',
    'iso_partner', # this is always the same as iso_parent, but easier to keep here for following maxing exercises
    'year',
    "income_tax_paid_on_cash_basis",
    "ln_income_tax_paid_on_cash_basis",
    "ln_profit_loss_before_income_tax_corrected",
    "ln_n_employees",
    "ln_unrelated_party_revenues",
    "ln_tangible_assets_except_cash",
    'ln_stated_capital',
    'ln_total_revenues',
    'ln_related_party_revenues',
    'ln_holding_or_managing_ip',
    "etr_foreign_corrected", # I only use foreign here, because countries with missing domestic data will not have domestic ETRs and we do not want to impute them either
    "cit",
    'ln_wage_monthly',
    'ln_gdp_current_usd',
    'ln_population',
    'ln_gvt_health_expenditure',
    'ln_n_companies_orbis',
    'ln_turnover_orbis',
    'ln_n_employees_orbis',
    'region_tjn',
    'g7',
    'g20',
    'eu27',
    'ukt'
]
domestic_imputation_sample = imputation_sample.loc[imputation_sample["iso_parent"] == imputation_sample["iso_partner"], domestic_variables]
domestic_imputation_sample.replace([np.inf, -np.inf], np.nan, inplace=True) 
foreign_imputation_sample = imputation_sample.loc[imputation_sample["iso_parent"] != imputation_sample["iso_partner"]]
foreign_imputation_sample.replace([np.inf, -np.inf], np.nan, inplace=True) 
true_values = cbcr_main_no_imputation_allsubgroupsonly[["iso_parent","iso_partner","year",'income_tax_paid_on_cash_basis','profit_loss_before_income_tax_corrected',
'n_employees',"unrelated_party_revenues","tangible_assets_except_cash",'stated_capital','total_revenues','related_party_revenues','holding_or_managing_ip']]

### 3.2 Impute values

#### 3.2.1 Define metaparameters for imputation

In [ ]:
# DEFINE SCORER TO VALIDATE AND COMPARE MODELS
scorer = make_scorer(r2_score)

In [ ]:
# For documentation purposes only: This is how the metaestimators were determined
if 0:
    def find_best_model(est,clf,df):
        
        for var in ["ln_profit_loss_before_income_tax_corrected",
        "ln_n_employees",
        "ln_unrelated_party_revenues",
        "ln_tangible_assets_except_cash",
        'ln_stated_capital',
        'ln_total_revenues',
        'ln_related_party_revenues',
        'ln_holding_or_managing_ip']:
            df = cbcr_main_no_imputation_allsubgroupsonly.copy().dropna(subset=["ln_profit_loss_before_income_tax_corrected",var])
            df[var].replace([np.inf, -np.inf], np.nan, inplace=True)
            print(var)
            y = df[var].values
            a = df.drop(columns=["income_tax_paid_on_cash_basis","ln_profit_loss_before_income_tax_corrected","ln_n_employees","ln_unrelated_party_revenues",
                "ln_tangible_assets_except_cash",'ln_stated_capital','ln_total_revenues','ln_related_party_revenues','ln_holding_or_managing_ip']
                +[_ for _ in df.columns if "fitted" in _]+[_ for _ in df.columns if "month_wage" in _])
            columns = np.array(a.columns)[a.dtypes!=object]
    #         print(columns)
            X = df[columns].values
    #         display(df[columns].describe())
            print(y.min(),y.mean(),y.max(),y.std())


            search = clf.fit(X,y)
            print(var, search.best_params_)
            try:
                plt.plot(search.param_grid["alpha"],search.cv_results_["mean_test_score"])
                plt.show()
            except:
                pass            

In [ ]:
# Estimate metaparameters
if 0: # For documentation purposes only: This is how the metaestimators were determined
    distributions = dict(learning_rate=np.linspace(0.01,0.2,7),
                            l2_regularization=np.logspace(1,3,11),
                            min_samples_leaf=[20],)
    est = HistGradientBoostingRegressor(random_state=0)

    clf = GridSearchCV(est, distributions, n_jobs=-1, cv=5,verbose=5, scoring=scorer)

    find_best_model(est,clf,foreign_imputation_sample.dropna(subset=["ln_profit_loss_before_income_tax_corrected"]))

In [ ]:
# Set identified parameters
metaparameters  = {
    "ln_profit_loss_before_income_tax_corrected": {'l2_regularization': 25.118864315095795, 'learning_rate': 0.105, 'min_samples_leaf': 20},
    "ln_n_employees": {'l2_regularization': 10.0, 'learning_rate': 0.1366666666666667, 'min_samples_leaf': 20},
    "ln_unrelated_party_revenues": {'l2_regularization': 10.0, 'learning_rate': 0.105, 'min_samples_leaf': 20},
    "ln_tangible_assets_except_cash": {'l2_regularization': 15.848931924611133, 'learning_rate': 0.07333333333333333, 'min_samples_leaf': 20}, # We take the metaparameters from ln_stated_capital as they are not estimated for ln_tangible_assets_except_cash
    'ln_stated_capital': {'l2_regularization': 15.848931924611133, 'learning_rate': 0.07333333333333333, 'min_samples_leaf': 20},
    'ln_total_revenues': {'l2_regularization': 39.810717055349734, 'learning_rate': 0.2, 'min_samples_leaf': 20},
    'ln_related_party_revenues': {'l2_regularization': 10.0, 'learning_rate': 0.07333333333333333, 'min_samples_leaf': 20},
    'ln_holding_or_managing_ip': {'l2_regularization': 10.0, 'learning_rate': 0.105, 'min_samples_leaf': 20},
}

#### 3.2.2 Define imputation function

In [ ]:
# DEFINE IMPUTING FUNCTION BASED ON BAYESIAN RIDGE
def impute(imputation_sample,columns_used_for_imputation,verbose=0):
    '''Imputes the variables specified by "columns" based on the data given in the "imputation_sample" (which also needs to include the variables specified in "columns")
      by BayesianRidge (which performs quite as well as missForest (R))'''
    imputation_sample = imputation_sample[imputation_sample['year'].between(first_year, first_year+n_years-1)] 
    imputation_sample = imputation_sample.dropna(axis=1, how='all')
    imputation_sample = imputation_sample.dropna(axis=0, how='all')
    
    values = imputation_sample[columns_used_for_imputation].values
    print(values.shape)
    estimator = BayesianRidge()                                                     #(n_estimators=10, random_state=0, bootstrap=True, n_jobs=-1)
    imp = IterativeImputer(max_iter=100, estimator = estimator,verbose=verbose) 
    imp.fit(values)
    new_values = imp.transform(values)
    print(new_values.shape)

    #Replace with full info
    for i,col in enumerate(columns_used_for_imputation):
        imputation_sample[col] = new_values[:, i]
    
    return imputation_sample

#### 3.2.3 Define imputation procedure for domestic values

In [ ]:
def fill_domestic(domestic_imputation_sample, bootstrap=False, output=False, columns=None):
    dfs = []  # list to store regression coefficient dataframes (if output=True)

    for var in ['ln_profit_loss_before_income_tax_corrected','ln_n_employees',"ln_unrelated_party_revenues",
        "ln_tangible_assets_except_cash",'ln_stated_capital','ln_total_revenues','ln_related_party_revenues','ln_holding_or_managing_ip']:
        
        # Initialize a linear regression model
        clf = LinearRegression()
        # Filter data based on the year range
        df = domestic_imputation_sample[domestic_imputation_sample['year'].between(first_year, first_year + n_years - 1)].copy()
                # Filter rows that have non-missing values for the current variable
        df = df.dropna(subset=[var])
        # Use default columns if none provided
        if columns is None:
            columns = ["ln_gdp_current_usd","ln_population","ln_gvt_health_expenditure","etr_foreign_corrected", 
                       "cit","ln_n_companies_orbis","ln_turnover_orbis","ln_n_employees_orbis" ,'ln_wage_monthly']
        
        # Bootstrap the data if specified
        if bootstrap:
            df = df[df["ln_n_companies_orbis"] > 0].sample(frac=1, replace=True).copy()
        
        # Get predictor variables and target variable
        X = df[df["ln_n_companies_orbis"] > 0][columns].values
        y = df[df["ln_n_companies_orbis"] > 0][var].values

        if X.shape[0] == 0:
            print(f"No samples available for training {var}. Skipping...")
            continue

        # Train the model
        clf.fit(X, y)

        # Print R-squared values if output flag is set to True
        if output:
            print(var, r2_score(y, clf.predict(X)))
            coeff_df = pd.DataFrame({'Variable': columns, 'Coefficient': clf.coef_})
            dfs.append(coeff_df.set_index('Variable'))

        # Make predictions and store them in the dataframe
        domestic_imputation_sample[f"fitted_{var}"] = clf.predict(domestic_imputation_sample[columns])
        
        # Special treatment for rows where "ln_n_companies_orbis" is 0
        domestic_imputation_sample.loc[domestic_imputation_sample["ln_n_companies_orbis"] == 0, f"fitted_{var}"] = 0
        
        # Convert predictions from logarithmic scale to linear scale
        domestic_imputation_sample[var[3:]] = np.exp(domestic_imputation_sample[f"fitted_{var}"]) - 1

    # Set the 'iso_partner' column equal to the 'iso_parent' column
    domestic_imputation_sample["iso_partner"] = domestic_imputation_sample["iso_parent"]

    # Display the regression coefficients if output flag is set to True
    if output:
        display(pd.concat(dfs, axis=1))

    return domestic_imputation_sample

In [ ]:
# To illustrate domestic imputation --> to figures notebook
variables_used_for_domestic_imputation = np.array(domestic_imputation_sample.columns)[domestic_imputation_sample.dtypes!=object]
variables_used_for_domestic_imputation = [_ for _ in variables_used_for_domestic_imputation if (_ not in true_values.columns) and (_[3:] not in true_values.columns)]
print(variables_used_for_domestic_imputation)
# ['etr_foreign_corrected', 'cit', 'ln_wage_monthly', 'ln_gdp_current_usd', 'ln_population', 'ln_gvt_health_expenditure', 'ln_n_companies_orbis', 'ln_turnover_orbis', 'ln_n_employees_orbis', 'g7', 'g20', 'eu27', 'ukt']
domestic_values_imputed = impute(domestic_imputation_sample,variables_used_for_domestic_imputation,verbose=5)
cbcr_with_imputed_domestic_values = fill_domestic(domestic_values_imputed,bootstrap=False,output=True,
                  columns =["ln_gdp_current_usd","ln_population","etr_foreign_corrected","ln_n_companies_orbis"])

### 3.2.4 Define imputation procedure for foreign values

In [ ]:
# Foreign profits: Use boosting
def fill_foreign(foreign_imputation_sample,bootstrap=False,output=False,n_repeats=2):
    i = 0
    if output:
        fig = plt.figure(figsize=(18,12))
    for var in ['ln_profit_loss_before_income_tax_corrected','ln_n_employees',"ln_unrelated_party_revenues","ln_tangible_assets_except_cash",
        'ln_stated_capital','ln_total_revenues','ln_related_party_revenues','ln_holding_or_managing_ip']:
        i += 1
        print(metaparameters.keys())
        clf = HistGradientBoostingRegressor(**metaparameters[var])

        foreign_imputation_sample = foreign_imputation_sample[foreign_imputation_sample['year'].between(first_year,first_year+n_years-1)]
        df = foreign_imputation_sample.dropna(subset=[var]).copy()
        
        a = df.drop(columns=["income_tax_paid_on_cash_basis",'ln_profit_loss_before_income_tax_corrected','ln_n_employees',
            "ln_unrelated_party_revenues","ln_tangible_assets_except_cash",'ln_stated_capital','ln_total_revenues',
            'ln_related_party_revenues','ln_holding_or_managing_ip']
            +[_ for _ in df.columns if "fitted" in _]+[_ for _ in df.columns if "wage_monthly" in _])
        columns = np.array(a.columns)[a.dtypes!=object]
#       print(columns)
        if bootstrap:
            df = df.sample(frac=1,replace=True)

        X = df[columns].values
        y = df[var].values

        clf.fit(X,y)
        foreign_imputation_sample.loc[:,"fitted_{}".format(var)] = clf.predict(foreign_imputation_sample[columns].values)
        

        if output:
            try:
                print([_ for _ in sorted(zip(clf.coef_,columns)) if _[0]!=0])
            except:
                pass
            result = permutation_importance(clf, X, y, n_repeats=n_repeats,
                                            random_state=42, n_jobs=1, scoring=scorer)
            sorted_idx = result.importances_mean.argsort()

            plt.subplot(1,3,i)
            sorted_idx = result.importances_mean.argsort()
            vals = 40
            labels = columns[sorted_idx][-vals:]
            labels = [_[-20:] for _ in labels] #cut if too long
            plt.boxplot(100*result.importances[sorted_idx].T[:,-vals:],
                       vert=False, labels=labels)
            plt.xlabel("Decrease in R2 (% points)")
            plt.title("{}".format(var))
            sns.despine(bottom=True,left=True)
            plt.xscale("log")
            #plt.xticks(rotation=90)

        
    if output:
        plt.tight_layout()
        plt.savefig(f"{output_figures}/permutation_estimation.pdf",bbox_inches="tight")
        #     plt.ylabel("Features")
        plt.show()
    return foreign_imputation_sample

In [ ]:
# Illustrate imputation of foreign values --> to figures notebook
cbcr_plus_imputed_foreign_values = fill_foreign(foreign_imputation_sample,bootstrap=False,output=False)

#### 3.2.5 Substitute imputed with correct values whenever we do not have correct values

In [ ]:
# To illustrate substitution of missing values
# Substitute imputed with correct values whenever we do not have correct values
for var in ['ln_profit_loss_before_income_tax_corrected','ln_n_employees',"ln_unrelated_party_revenues","ln_tangible_assets_except_cash",
  'ln_stated_capital','ln_total_revenues','ln_related_party_revenues','ln_holding_or_managing_ip']:
    cbcr_with_imputed_foreign_values_countrygroups = cbcr_plus_imputed_foreign_values.copy()
    cbcr_with_imputed_foreign_values_countrygroups.loc[~np.isnan(cbcr_with_imputed_foreign_values_countrygroups[var]),"fitted_"+var
                                         ] = cbcr_with_imputed_foreign_values_countrygroups.loc[~np.isnan(cbcr_with_imputed_foreign_values_countrygroups[var]),var] 
    cbcr_with_imputed_domestic_values.loc[~np.isnan(cbcr_with_imputed_domestic_values[var]),"fitted_"+var
                                          ] = cbcr_with_imputed_domestic_values.loc[~np.isnan(cbcr_with_imputed_domestic_values[var]),var] 
  
    cbcr_with_imputed_domestic_values.loc[(cbcr_with_imputed_domestic_values["ln_n_companies_orbis"]==0),"fitted_"+var] = 0
    cbcr_with_imputed_foreign_values_countrygroups.loc[cbcr_with_imputed_foreign_values_countrygroups["ln_n_companies_orbis"]==0,"fitted_"+var] = 0

### 3.3 Define how values of country groups should be assigned to individual countries

In [ ]:
OAF = AFRIC = set(
    cbcr_main_no_imputation_allsubgroupsonly.loc[cbcr_main_no_imputation_allsubgroupsonly["region_tjn"] == "Africa", "iso_partner"].dropna()
    )
OAS = ASIAT = set(
    cbcr_main_no_imputation_allsubgroupsonly.loc[cbcr_main_no_imputation_allsubgroupsonly["region_tjn"] == "Asia", "iso_partner"].dropna()
    )
OTE = EUROP = set(
    cbcr_main_no_imputation_allsubgroupsonly.loc[cbcr_main_no_imputation_allsubgroupsonly["region_tjn"] == "Europe", "iso_partner"].dropna()
    )
OAM = LAC_sca = set(
    cbcr_main_no_imputation_allsubgroupsonly.loc[cbcr_main_no_imputation_allsubgroupsonly["region_tjn"].str.contains("America", na=False), "iso_partner"].dropna()
)
GRPS = set(cbcr_main_no_imputation_allsubgroupsonly["iso_partner"].dropna())
LAC = set(
    cbcr_main_no_imputation_allsubgroupsonly.loc[cbcr_main_no_imputation_allsubgroupsonly["region_tjn"] == "Latin America and the Caribbean", "iso_partner"].dropna()
    )

d_groups2countries = {"OAF": OAF, "AFRIC": AFRIC, "OAS": OAS, "ASIAT": ASIAT, "OTE": OTE, "EUROP": EUROP, "OAM": OAM, "LAC_sca": LAC_sca, "GRPS": GRPS, "LAC": LAC}

In [ ]:
# function to adjust the total of country groups to a known total
def adjust_total(country, data, year):
    '''Adjust the total of country groups to a known total for a given year'''
    data = data[data['year'] == year]
    data = pd.DataFrame(data)
    
    agg_data = true_values.loc[(true_values["iso_parent"] == country) & (true_values["year"] == year)].copy()
    
    if len(agg_data) == 0:
        return data
    
    if country in ["AUT","SWE","NOR"]:
        agg_data["iso_partner"] = agg_data["iso_partner"].replace("LAC","LAC_sca")
    agg_data = agg_data.set_index("iso_partner")
    

    cs_included = list(agg_data.index)
    for region in ["OAF","OAM","OAS","OTE","AFRIC","ASIAT","EUROP","LAC","LAC_sca","FJT","GRPS"]:
        if region in agg_data.index:
            
            if region == "GRPS":
                cs_group = d_groups2countries[region] - set(cs_included)
                truth_remove = 0
            elif (region == "FJT"): 
                if ("GRPS" in agg_data.index):
                    continue #otherwise fix it with GRPS
                #Countries in the main df but not in the original one and not fixed before = FJT
                cs_group = set(data["iso_partner"])- set(cs_included)
#                 print(cs_group)
                #Remove from FJT everything fixed before
                truth_remove = agg_data.loc[list(set(agg_data.index) - set(["FJT",country])),
                ['profit_loss_before_income_tax_corrected','n_employees',"unrelated_party_revenues","tangible_assets_except_cash",
                'stated_capital','total_revenues','related_party_revenues','holding_or_managing_ip']].sum().fillna(0)

            else:
                cs_included += list(d_groups2countries[region])
                cs_group = d_groups2countries[region] - set(agg_data.index)
                truth_remove = 0

            truth = agg_data.loc[region,[
                'profit_loss_before_income_tax_corrected','n_employees',
                "unrelated_party_revenues","tangible_assets_except_cash",
                'stated_capital','total_revenues','related_party_revenues',
                'holding_or_managing_ip']] - truth_remove
            if isinstance(truth, pd.DataFrame):
                truth = truth.sum()
    
            pred = np.exp(data.loc[data["iso_partner"].isin(cs_group),[
                'fitted_ln_profit_loss_before_income_tax_corrected','fitted_ln_n_employees',
                "fitted_ln_unrelated_party_revenues","fitted_ln_tangible_assets_except_cash",
                'fitted_ln_stated_capital','fitted_ln_total_revenues','fitted_ln_related_party_revenues',
                'fitted_ln_holding_or_managing_ip']])-1
            
            #If profits are negative in the region set them up to 0
            scale_by = pred.sum().values/truth.values
            scale_by[truth.values<0] = np.inf
            
            aux = 1+pred/scale_by
            data.loc[data["iso_partner"].isin(cs_group),
            ['fitted_ln_profit_loss_before_income_tax_corrected','fitted_ln_n_employees',
            "fitted_ln_unrelated_party_revenues","fitted_ln_tangible_assets_except_cash",
            'fitted_ln_stated_capital','fitted_ln_total_revenues','fitted_ln_related_party_revenues',
            'fitted_ln_holding_or_managing_ip']] = np.log(aux.apply(pd.to_numeric))

        else: #Countries where no operations
            if region == "GRPS":
                cs_group = d_groups2countries[region] - set(cs_included)
                
                data.loc[data["iso_partner"].isin(cs_group),
                ['fitted_ln_profit_loss_before_income_tax_corrected','fitted_ln_n_employees',"fitted_ln_unrelated_party_revenues","fitted_ln_tangible_assets_except_cash",
                    'fitted_ln_stated_capital','fitted_ln_total_revenues','fitted_ln_related_party_revenues','fitted_ln_holding_or_managing_ip']] = 0
                                
    return data

In [ ]:
# To illustrate final datasets
cbcr_with_imputed_foreign_values_grouped = [
    adjust_total(country, group_data, year) 
    for (country, year), group_data in cbcr_with_imputed_foreign_values_countrygroups.groupby(["iso_parent", "year"])
]
cbcr_with_imputed_foreign_values = pd.concat(cbcr_with_imputed_foreign_values_grouped)

for var in ['ln_profit_loss_before_income_tax_corrected','ln_n_employees',"ln_unrelated_party_revenues","ln_tangible_assets_except_cash"]:
    cbcr_with_imputed_foreign_values.loc[:,var[3:]] = np.exp(cbcr_with_imputed_foreign_values["fitted_" + var])-1
    cbcr_with_imputed_domestic_values.loc[:,var[3:]] = np.exp(cbcr_with_imputed_domestic_values["fitted_" + var])-1

### 3.4 Check validity of imputation procedure

In [ ]:
# DOMESTIC RESULTS
for c,x,y in zip(cbcr_with_imputed_domestic_values["iso_parent"],np.exp(cbcr_with_imputed_domestic_values["ln_n_companies_orbis"]),np.exp(cbcr_with_imputed_domestic_values["fitted_ln_profit_loss_before_income_tax_corrected"])):
    if c in tax_havens:
        plt.annotate(c,(x,y))
plt.plot(np.exp(cbcr_with_imputed_domestic_values["ln_n_companies_orbis"]),np.exp(cbcr_with_imputed_domestic_values["fitted_ln_profit_loss_before_income_tax_corrected"]),".",color="grey")
plt.plot(np.exp(cbcr_with_imputed_domestic_values["ln_n_companies_orbis"]),np.exp(cbcr_with_imputed_domestic_values["ln_profit_loss_before_income_tax_corrected"]),".",ms=8)
plt.xscale("log")
plt.yscale("log")
sns.despine(bottom=True,left=True)
plt.xlabel("Number of companies (expected in Orbis)")
plt.ylabel("Total profits (fitted in grey, in data in blue)")
plt.ylim(1E7,2E12)

In [ ]:
# Function to check how well our imputation works
def plot_matching_foreign(true_values,foreign_imputation_sample,annot=False):
    #What we get vs what we know
    #x_fjt = true_values.loc[true_values["iso3_d"]=="FJT"].groupby("iso3_o").agg(lambda x: np.sum(x[x>0]))
    numeric_cols = true_values.select_dtypes(include=np.number).columns
    x_fjt = true_values.loc[true_values["iso_partner"] == "FJT"].groupby("iso_parent")[numeric_cols].agg(lambda x: np.sum(x[x > 0]))
    x_other = true_values.loc[
        (true_values["iso_parent"] != true_values["iso_partner"])
        & (~true_values["iso_parent"].isin(x_fjt.index))
        ].groupby("iso_parent")[numeric_cols].agg(lambda x: np.sum(x[x > 0]))
    x = pd.concat([x_fjt,x_other])
    a = foreign_imputation_sample.groupby("iso_parent")["fitted_ln_profit_loss_before_income_tax_corrected"].agg(lambda x: np.sum(np.exp(x[x>0]))).to_dict()
    x["a"] = x.index.map(a)
    plt.plot(x["profit_loss_before_income_tax_corrected"],x["a"],"o")
    plt.plot([1E8,1E12],[1E8,1E12],"--",color="grey")
    plt.xscale("log")
    plt.yscale("log")
    if annot:
        for x_,y_,c_ in zip(x["profit_loss_before_income_tax_corrected"],x["a"],x.index):
            if (x_>0) and (y_>0): 
                plt.annotate(c_,(x_,y_))

    plt.xlabel("Real foreign profits")
    plt.ylabel("Predicted foreign profits")
    sns.despine(bottom=True,left=True)
    plt.show()

In [ ]:
ground = true_values.loc[true_values["iso_parent"]=="NLD"]
for var in ['profit_loss_before_income_tax_corrected','n_employees',"unrelated_party_revenues","tangible_assets_except_cash",'stated_capital','total_revenues','related_party_revenues','holding_or_managing_ip']:
    ground["ln_"+var] = np.log(1+ground[var])
  
print("Predictions for NLDs")
display(ground)
display(np.exp(cbcr_with_imputed_domestic_values.loc[cbcr_with_imputed_domestic_values["iso_parent"]=="NLD",
                                                     ['fitted_ln_profit_loss_before_income_tax_corrected','fitted_ln_n_employees',
                                                      "fitted_ln_unrelated_party_revenues","fitted_ln_tangible_assets_except_cash",
                                                      'ln_profit_loss_before_income_tax_corrected','ln_n_employees',
                                                      "ln_unrelated_party_revenues","ln_tangible_assets_except_cash"]]))
foreign_nl = cbcr_with_imputed_foreign_values.loc[cbcr_with_imputed_foreign_values["iso_parent"]=="NLD",
                                                    ['fitted_ln_profit_loss_before_income_tax_corrected','fitted_ln_n_employees',
                                                      "fitted_ln_unrelated_party_revenues","fitted_ln_tangible_assets_except_cash"]]
display(np.exp(foreign_nl).sum().T)

plot_matching_foreign(true_values,cbcr_with_imputed_foreign_values, annot=True)

### 3.5 Generate bootstrapped sample with imputed values

In [ ]:
def one_boot():
    '''Fit the modelRequires as global variables:    - df_domestic    - df_foreign    - df_fin_mis    - df_dom_remaining'''
    ## DOMESTIC: Impute domestic data and fit model
    variables_used_for_domestic_imputation = np.array(domestic_imputation_sample.columns)[domestic_imputation_sample.dtypes!=object]
    variables_used_for_domestic_imputation = [_ for _ in variables_used_for_domestic_imputation if (_ not in true_values.columns) and (_[3:] not in true_values.columns)]

    imputation_domestic = impute(domestic_imputation_sample,variables_used_for_domestic_imputation)
    cbcr_domestic_filled = fill_domestic(imputation_domestic,bootstrap=True,output=True)

    ## FOREIGN: 
    # Impute foreign data and fit model
    cbcr_foreign_filled = fill_foreign(foreign_imputation_sample,bootstrap=True,output=False)

    
    # Correct the information that we know from the raw data
    for var in ['ln_profit_loss_before_income_tax_corrected','ln_n_employees',"ln_unrelated_party_revenues","ln_tangible_assets_except_cash",
        'ln_stated_capital','ln_total_revenues','ln_related_party_revenues','ln_holding_or_managing_ip']:
                     #cbcr_foreign_filled[var+'imputed'] = 1
                    # cbcr_domestic_filled[var+'imputed'] = 1
                     cbcr_foreign_filled.loc[~np.isnan(cbcr_foreign_filled[var]),"fitted_"+var] = cbcr_foreign_filled.loc[~np.isnan(cbcr_foreign_filled[var]),var]
                     #cbcr_foreign_filled.loc[~np.isnan(cbcr_foreign_filled[var]), var+'imputed'] = 0
                     cbcr_domestic_filled.loc[~np.isnan(cbcr_domestic_filled[var]),"fitted_"+var] = cbcr_domestic_filled.loc[~np.isnan(cbcr_domestic_filled[var]),var] 
                     #cbcr_domestic_filled.loc[~np.isnan(cbcr_domestic_filled[var]), var+'imputed'] = 0

    #countries with no companies have no profits/etc
    cbcr_domestic_filled.loc[(cbcr_domestic_filled["ln_n_companies_orbis"]==0),"fitted_"+var] = 0
    cbcr_foreign_filled.loc[cbcr_foreign_filled["ln_n_companies_orbis"]==0,"fitted_"+var] = 0

    # Placeholder for the results
    all_data = []
    
    # Loop through each iso_parent group
    for group, group_data in cbcr_foreign_filled.groupby("iso_parent"):
        # Adjust totals year by year for each iso_parent
        for year in range(first_year, first_year + n_years):
            yearly_data = adjust_total(group, group_data, year)
            all_data.append(yearly_data)

    cbcr_foreign_filled = pd.concat(all_data)

    ## CREATE DATASET
    # Calculate the numbers from the logs
    for var in ['ln_profit_loss_before_income_tax_corrected','ln_n_employees',"ln_unrelated_party_revenues","ln_tangible_assets_except_cash",
        'ln_stated_capital','ln_total_revenues','ln_related_party_revenues','ln_holding_or_managing_ip']:
        cbcr_foreign_filled.loc[:,var[3:]] = np.exp(cbcr_foreign_filled["fitted_"+var])-1
        cbcr_domestic_filled.loc[:,var[3:]] = np.exp(cbcr_domestic_filled["fitted_"+var])-1

    # Use only the domestic data of the countries we have no information
    cbcr_domestic_remaining = cbcr_domestic_filled.loc[~cbcr_domestic_filled["iso_parent"].isin(true_values["iso_parent"].unique())]

    # Merge the domestic data to the raw data
    cbcr_main = pd.concat([true_values,cbcr_domestic_remaining],sort=False)
    
    # Specify which raws correspond to domestic data
    cbcr_main["Domestic"] = (cbcr_main["iso_parent"]==cbcr_main["iso_partner"]).astype(int)

    
    ## ADJUST DATASET
    # Calculate the sum by destination country in the raw data
    cbcr_main_withsum = cbcr_main.copy()
    for var in ['n_employees',"unrelated_party_revenues","tangible_assets_except_cash",
        	'stated_capital','total_revenues','related_party_revenues','holding_or_managing_ip']:
        cbcr_main_withsum[f"raw_{var}_sum"] = cbcr_main_withsum.groupby(["iso_partner","Domestic"])[var].transform(np.sum)
        cbcr_main_withsum[f"{var}_pred"] = cbcr_main_withsum[f"{var}"].copy()
    
    # Calculate the sum by destination country in the predicted data
    predicted_values = cbcr_foreign_filled.groupby("iso_partner")[['n_employees',"unrelated_party_revenues","tangible_assets_except_cash",
        'stated_capital','total_revenues','related_party_revenues','holding_or_managing_ip']].sum()
    predicted_values.columns = ["n_employees_pred_sum","unrelated_party_revenues_pred_sum","tangible_assets_except_cash_pred_sum",
        'stated_capital_pred_sum','total_revenues_pred_sum','related_party_revenues_pred_sum','holding_or_managing_ip_pred_sum']
    
    # Merge both to adjust
    predicted_values = pd.merge(cbcr_main_withsum,predicted_values.reset_index(),how="outer")   
    
    # Calculate the ratio between predicted and raw, and keep the lowest of the two values
    predicted_values["extra_n_employees"] = (predicted_values["n_employees_pred_sum"]+1)/(predicted_values["raw_n_employees_sum"]+1).round(2)
    predicted_values["extra_unrelated_party_revenues"] = (predicted_values["unrelated_party_revenues_pred_sum"]+1)/(predicted_values["raw_unrelated_party_revenues_sum"]+1).round(2)
    predicted_values["extra_tangible_assets_except_cash"] = (predicted_values["tangible_assets_except_cash_pred_sum"]+1)/(predicted_values["raw_tangible_assets_except_cash_sum"]+1).round(2)
    predicted_values["extra_stated_capital"] = (predicted_values["stated_capital_pred_sum"]+1)/(predicted_values["raw_stated_capital_sum"]+1).round(2)
    predicted_values["extra_total_revenues"] = (predicted_values["total_revenues_pred_sum"]+1)/(predicted_values["raw_total_revenues_sum"]+1).round(2)
    predicted_values["extra_related_party_revenues"] = (predicted_values["related_party_revenues_pred_sum"]+1)/(predicted_values["raw_related_party_revenues_sum"]+1).round(2)
    predicted_values["extra_holding_or_managing_ip"] = (predicted_values["holding_or_managing_ip_pred_sum"]+1)/(predicted_values["raw_holding_or_managing_ip_sum"]+1).round(2)
    predicted_values["one"] = 1
    predicted_values["extra_top"] = predicted_values[["one","extra_n_employees","extra_unrelated_party_revenues","extra_tangible_assets_except_cash",
        "extra_stated_capital","extra_total_revenues","extra_related_party_revenues","extra_holding_or_managing_ip"]].max(1)
    predicted_values["extra_max"] = predicted_values[["extra_n_employees","extra_unrelated_party_revenues","extra_tangible_assets_except_cash",
        "extra_stated_capital","extra_total_revenues","extra_related_party_revenues","extra_holding_or_managing_ip"]].max(1)    
    predicted_values["extra"] = predicted_values[["extra_n_employees","extra_unrelated_party_revenues","extra_tangible_assets_except_cash",
        "extra_stated_capital","extra_total_revenues","extra_related_party_revenues","extra_holding_or_managing_ip"]].min(1)
    # If max is <1 don't adjust
    predicted_values.loc[predicted_values['extra_top']==1,"extra"] = predicted_values.loc[predicted_values['extra_top']==1,"extra_max"]
    # If max is >1 and min <1
    predicted_values.loc[(predicted_values['extra_max']>1)&(predicted_values['extra']<1),"extra"] = 1
    # Cap the adjustment to 10
    predicted_values.loc[predicted_values["extra"]>10,"extra"] = 10 #np.nan#Capped at 10 times 10# np.nan #Will remove that observation (during the correction step) when we are not capturing almost anything in the data
    # Don't adjust domestic data    
    predicted_values.loc[predicted_values["Domestic"]==1,"extra"] = 1
    
    # Perform adjustment
    predicted_values["profit_loss_before_income_tax_corrected"] = predicted_values["profit_loss_before_income_tax_corrected"]*predicted_values["extra"]
    for var in ['n_employees',"unrelated_party_revenues","tangible_assets_except_cash",'stated_capital','total_revenues','related_party_revenues','holding_or_managing_ip']:
        predicted_values[f"{var}_pred"] *= predicted_values["extra"]
    
    # Return data
    return predicted_values[["iso_partner","iso_parent","year","Domestic","income_tax_paid_on_cash_basis","n_employees_pred","unrelated_party_revenues_pred","tangible_assets_except_cash_pred",
    "profit_loss_before_income_tax_corrected","extra","n_employees","unrelated_party_revenues","tangible_assets_except_cash","profit_loss_before_income_tax_corrected",
    'stated_capital','total_revenues','related_party_revenues','holding_or_managing_ip']]

In [ ]:
# Generate bootstrapped sample, (bootstrapping specified in "range")
with np.errstate(divide='ignore', invalid='ignore'), open(f'{data_intermediate}/cbcr_bootstrapped_imputation.csv',"w+") as f:
    for i in range(1000):
        print(i+1, end=": ")
        pred = one_boot()
        pred["n_rep"] = i
        pred.to_csv(f,sep="\t",index = None,header = i==0)

### Combine imputed data with other data

In [ ]:
def num(x):
    """
    Convert numbers of strings to floats
    """
    try:
        return float(x)
    except:
        return np.nan

In [ ]:
cbcr_with_imputeded_data = pd.read_csv(f'{data_intermediate}/cbcr_bootstrapped_imputation-Alison.csv',sep="\t").reset_index(drop=True)
# In the following, I only use half of the imputed datasets as my computer cannot process the data otherwise
cbcr_with_imputeded_data = cbcr_with_imputeded_data[cbcr_with_imputeded_data['n_rep'] < 500]

#cbcr_main = cbcr_main.drop_duplicates()
for col in list(cbcr_with_imputeded_data.columns)[2:]:
    cbcr_with_imputeded_data[col] = cbcr_with_imputeded_data[col].apply(num)
    
#cbcr_main = cbcr_main.dropna()
columns_from_original_data = ['iso_parent','iso_partner','year',
            'wage_monthly','gdp_current_usd','gvt_health_expenditure',
            'cit','etr_foreign_corrected','population', 'etr_domestic_corrected', 'etr_average_corrected',
            'region_tjn', 'g7', 'eu27', 'g20', 'ukt']
cbcr_with_imputeded_data = cbcr_with_imputeded_data.merge(cbcr_main_no_imputation_allsubgroupsonly[columns_from_original_data],
            on=['iso_partner', 'year'], how='outer')
cbcr_with_imputeded_data['payroll'] = cbcr_with_imputeded_data['n_employees'] * cbcr_with_imputeded_data['wage_monthly'] * 12 
cbcr_with_imputeded_data.to_csv(f'{data_final}/cbcr_main.csv')